In [ ]:
%load_ext autoreload
%autoreload 2
# imports 
import h5py
import pandas as pd
import numpy as np
import corner
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal, norm, truncnorm
import sys
sys.path.insert(0, '/Users/smericks/Desktop/StrongLensing/padma_timedelays/fasttdc/')
import Utils.make_data_vectors as mdv

Step #2: Create data vectors from fermat potential diff. samples, send to Sherlock for galkin computations

In [ ]:
# set the anistropy "modeling prior" here
BETA_ANI_MU = 0.
BETA_ANI_SIGMA = 0.2 

In [ ]:
# load in NPE models
images_path = 'DataVectors/hst_image_models.h5'
h5 = h5py.File(images_path, 'r')

npe_models = {}

npe_models['mu_npe'] = h5['mu_npe'][:]
npe_models['cov_npe'] = h5['cov_npe'][:]
npe_models['catalog_idx'] = h5['catalog_idx'][:]
h5.close()

# load in ground truth
df_path = 'DataVectors/truth_metadata.csv'
truth_df = pd.read_csv(df_path)
npe_models['truth_df'] = truth_df

In [ ]:
# generate fpd samples

metadata_df = npe_models['truth_df']

dbls_idxs = np.where(metadata_df.loc[:,'point_source_parameters_num_images'].to_numpy() == 2.)[0]
dbls_catalog_idxs = metadata_df.loc[dbls_idxs,'catalog_idx'].to_numpy()
quads_idxs = np.where(metadata_df.loc[:,'point_source_parameters_num_images'].to_numpy() == 4.)[0]
quads_catalog_idxs = metadata_df.loc[quads_idxs,'catalog_idx'].to_numpy()

# NOTE: hardcodings in this function for doubles and quads idxs
NUM_FPD_SAMPS = int(500) 

# helper function
def fpd_samps_helper(metadata_df,npe_mu,npe_cov,num_images):

    if num_images == 2:
        return mdv.fpd_gamma_samples(
            x_im=metadata_df.loc[dbls_idxs,
                ['point_source_parameters_x_image_0',
                'point_source_parameters_x_image_1']].to_numpy(),
            y_im=metadata_df.loc[dbls_idxs,
                ['point_source_parameters_y_image_0',
                'point_source_parameters_y_image_1']].to_numpy(),
            npe_mu=npe_mu[dbls_idxs],npe_cov=npe_cov[dbls_idxs],
            num_fpd_samps=NUM_FPD_SAMPS)
    
    elif num_images == 4:
        return mdv.fpd_gamma_samples(
            x_im=metadata_df.loc[quads_idxs,
                ['point_source_parameters_x_image_0',
                'point_source_parameters_x_image_1',
                'point_source_parameters_x_image_2',
                'point_source_parameters_x_image_3']].to_numpy(),
            y_im=metadata_df.loc[quads_idxs,
                ['point_source_parameters_y_image_0',
                'point_source_parameters_y_image_1',
                'point_source_parameters_y_image_2',
                'point_source_parameters_y_image_3']].to_numpy(),
            npe_mu=npe_mu[quads_idxs],npe_cov=npe_cov[quads_idxs],
            num_fpd_samps=NUM_FPD_SAMPS)
    
    else:
        raise ValueError("Num images not supported")

# produce them here!!
# quads
fpd_samps_quads, lens_param_samps_quads = fpd_samps_helper(
    metadata_df,npe_models['mu_npe'],npe_models['cov_npe'],num_images=4.)
# doubles
fpd_samps_dbls, lens_param_samps_dbls = fpd_samps_helper(
    metadata_df,npe_models['mu_npe'],npe_models['cov_npe'],num_images=2.)

In [ ]:
# function for de-biasing the fpd / lens_param samps
def debiased_samples(fpd_samps,lens_param_samps,catalog_idxs,truth_df,
    change_precision_JWST=False,change_precision_TDCOSMO25=False):

    # initialize arrays
    debiased_fpd_samps = np.empty(np.shape(fpd_samps))
    debiased_lens_param_samps = np.empty(np.shape(lens_param_samps))

    for lens_idx in range(0,len(catalog_idxs)):

        my_samps = np.concatenate((fpd_samps[lens_idx],lens_param_samps[lens_idx]),axis=1)
        # TODO: make sure this indexing works
        truth_df_row = truth_df[truth_df['catalog_idx'] == catalog_idxs[lens_idx]]
        if np.shape(fpd_samps)[-1] == 3:
            gt_for_debiasing = truth_df_row[['fpd01','fpd02','fpd03',
                'main_deflector_parameters_theta_E','main_deflector_parameters_gamma1',
                'main_deflector_parameters_gamma2','main_deflector_parameters_gamma',
                'main_deflector_parameters_e1','main_deflector_parameters_e2',
                'main_deflector_parameters_center_x','main_deflector_parameters_center_y',
                'source_parameters_center_x','source_parameters_center_y']].to_numpy()[0]
        elif np.shape(fpd_samps)[-1] ==1:
            gt_for_debiasing = truth_df_row[['fpd01',
                'main_deflector_parameters_theta_E','main_deflector_parameters_gamma1',
                'main_deflector_parameters_gamma2','main_deflector_parameters_gamma',
                'main_deflector_parameters_e1','main_deflector_parameters_e2',
                'main_deflector_parameters_center_x','main_deflector_parameters_center_y',
                'source_parameters_center_x','source_parameters_center_y']].to_numpy()[0]
            
        # starting point
        Mu = np.mean(my_samps,axis=0)
        Cov = np.cov(my_samps,rowvar=False)

        if change_precision_JWST:
            # change precision of the Cov s.t. sigma(fpd01) is 2% (Williams)
            desired_fpd_std_dev = 0.02 * gt_for_debiasing[0]
            change_precision_factor = desired_fpd_std_dev / np.sqrt(Cov[0,0]) 
            Cov *= (change_precision_factor**2)

        if change_precision_TDCOSMO25:
            desired_gamma_lens_std_dev = 0.04
            change_precision_factor = desired_gamma_lens_std_dev / np.sqrt(Cov[-7,-7]) 
            Cov *= (change_precision_factor**2)

        # de-bias with this line
        if gt_for_debiasing is not None:
            Mu = multivariate_normal.rvs(mean=gt_for_debiasing,cov=Cov)

        num_gaussian_samps = np.shape(fpd_samps)[1]
        gaussianized_samps = multivariate_normal.rvs(mean=Mu,cov=Cov,
            size=num_gaussian_samps)
        
        # TODO: separate into fpds and lens params
        num_fpd = np.shape(fpd_samps)[-1]
        debiased_fpd_samps[lens_idx] = gaussianized_samps[:,:num_fpd]
        debiased_lens_param_samps[lens_idx] = gaussianized_samps[:,num_fpd:]

    return debiased_fpd_samps, debiased_lens_param_samps, catalog_idxs

In [ ]:
len(dbls_catalog_idxs)

In [ ]:
input_datasets = [
    [fpd_samps_quads,lens_param_samps_quads,quads_catalog_idxs,metadata_df],
    [fpd_samps_dbls,lens_param_samps_dbls,dbls_catalog_idxs,metadata_df],
]

output_h5_files = [
    'DataVectors/quad_posteriors_JWST_DEBIASED.h5',
    'DataVectors/dbl_posteriors_JWST_DEBIASED.h5',
]

for i in range(0,len(input_datasets)):

    debiased_fpd, debiased_lp, catalog_idxs = debiased_samples(
        input_datasets[i][0],input_datasets[i][1],
        input_datasets[i][2],input_datasets[i][3],
        change_precision_JWST=True)

    # FIRST: generate some beta_ani samps!
    # truncate at beta_ani = -1...
    beta_ani_samps = truncnorm.rvs(
        (-0.5 - BETA_ANI_MU)/BETA_ANI_SIGMA, # lower truncation at -0.5
        (0.5 - BETA_ANI_MU)/BETA_ANI_SIGMA, # upper truncation at 0.5
        loc=BETA_ANI_MU,scale=BETA_ANI_SIGMA,
        size=(debiased_fpd.shape[0],debiased_fpd.shape[1]))

    # now, write the .h5 file and store all the samps...
    h5f = h5py.File(output_h5_files[i], 'w')
    h5f.create_dataset('catalog_idxs', data=catalog_idxs)
    h5f.create_dataset('fpd_samps', data=debiased_fpd)
    h5f.create_dataset('lens_param_samps', data=debiased_lp)
    h5f.create_dataset('beta_ani_samps', data=beta_ani_samps)

    # initialize empty kinematics!!
    h5f.create_dataset('c_sqrtJ_samps', data=-1*np.ones(
        (debiased_fpd.shape[0],debiased_fpd.shape[1],1)))
    # initialize empty kinematics??
    h5f.create_dataset('MUSE_c_sqrtJ_samps', data=-1*np.ones(
        (debiased_fpd.shape[0],debiased_fpd.shape[1],3))) # 3 bins for MUSE
    h5f.create_dataset('NIRSPEC_c_sqrtJ_samps', data=-1*np.ones(
        (debiased_fpd.shape[0],debiased_fpd.shape[1],10))) # 10 bins for NIRSPEC
    
    h5f.close()

let's make sure everything worked alright ...

In [ ]:
# ---- CHOOSE WHICH FILE TO INSPECT ----
h5_file_to_check = 'DataVectors/dbl_posteriors_TDCOSMO25_DEBIASED.h5'
lens_idx = 2   # which lens (row) to plot

# ---- READ BACK THE FILE ----
with h5py.File(h5_file_to_check, 'r') as h5f:
    print("Keys in file:", list(h5f.keys()))
    catalog_idxs      = h5f['catalog_idxs'][:]
    debiased_fpd      = h5f['fpd_samps'][:]
    debiased_lp       = h5f['lens_param_samps'][:]
    beta_ani_samps    = h5f['beta_ani_samps'][:]
    c_sqrtJ_samps     = h5f['c_sqrtJ_samps'][:]
    MUSE_c_sqrtJ      = h5f['MUSE_c_sqrtJ_samps'][:]
    NIRSPEC_c_sqrtJ   = h5f['NIRSPEC_c_sqrtJ_samps'][:]

# ---- SANITY CHECKS ----
print("catalog_idxs shape:   ", catalog_idxs.shape)
print("fpd_samps shape:      ", debiased_fpd.shape)
print("lens_param_samps shape:", debiased_lp.shape)
print("beta_ani_samps shape: ", beta_ani_samps.shape)
print("c_sqrtJ_samps shape:  ", c_sqrtJ_samps.shape)
print("MUSE shape:           ", MUSE_c_sqrtJ.shape)
print("NIRSPEC shape:        ", NIRSPEC_c_sqrtJ.shape)

# ---- LABELS ----
labels = ['fpd01','theta_E','gamma1','gamma2',
          'gamma_lens','e1','e2','x_lens','y_lens','x_src','y_src']

# ---- BUILD GROUND-TRUTH VECTOR FROM metadata_df ----
# grab the catalog index of the lens we're plotting
this_catalog_idx = catalog_idxs[lens_idx]
if isinstance(this_catalog_idx, bytes):
    this_catalog_idx = this_catalog_idx.decode('utf-8')
print('this_catalog_idx: ', this_catalog_idx)
gt_row = metadata_df[metadata_df['catalog_idx'] == this_catalog_idx].iloc[0]

# NOTE: fpd01/02/03 are usually NOT stored directly in metadata_df,
# so we set them to None (no line drawn). Adjust names below to match
# your metadata_df column names exactly.
truths = [
    gt_row['fpd01'],  # fpd01
    gt_row['main_deflector_parameters_theta_E'],
    gt_row['main_deflector_parameters_gamma1'],
    gt_row['main_deflector_parameters_gamma2'],
    gt_row['main_deflector_parameters_gamma'],
    gt_row['main_deflector_parameters_e1'],
    gt_row['main_deflector_parameters_e2'],
    gt_row['main_deflector_parameters_center_x'],
    gt_row['main_deflector_parameters_center_y'],
    gt_row['source_parameters_center_x'],
    gt_row['source_parameters_center_y'],
]

# ---- PLOT ----
my_samps = np.concatenate(
    (debiased_fpd[lens_idx], debiased_lp[lens_idx]), axis=1)

figure = corner.corner(
    my_samps, plot_datapoints=False,
    color='salmon', levels=[0.68, 0.95], fill_contours=True,
    labels=labels, dpi=300, fig=None,
    label_kwargs={'fontsize': 24}, smooth=0.9,
    truths=truths, truth_color='black')

figure.suptitle(f'lens_idx={lens_idx}, catalog_idx={this_catalog_idx}',
                fontsize=20)
plt.show()